In [10]:
# ---- To test randomly later -----

In [1]:
# Cell 1 — bootstrap + imports
import sys
import re
import random
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from fineas.io.paths import (
    raw_yf_ohlcv_part_path,
    norm_ohlcv_part_path,
)
from fineas.io.qc import (
    qc_raw_ohlcv_partition,
    qc_norm_ohlcv_partition,
)

In [2]:
# Cell 2 — parameterised selector (explicit OR random existing partition)
INTERVAL = "1d"

# Choose one mode:
USE_RANDOM_EXISTING = True

# If USE_RANDOM_EXISTING = False, set these:
SYMBOL = "AAPL"
YEAR = 2022
MONTH = 1

raw_interval_root = ROOT / "data" / "raw" / "yfinance" / "ohlcv" / f"interval={INTERVAL}"

def _parse_symbol_year_month(p: Path):
    s = str(p).replace("\\", "/")
    m = re.search(r"symbol=([^/]+)/year=(\d{4})/month=(\d{2})/part-(\d{4})-(\d{2})\.parquet$", s)
    if not m:
        raise ValueError(f"could not parse partition from path: {p}")
    sym = m.group(1)
    y = int(m.group(2))
    mo = int(m.group(3))
    return sym, y, mo

if USE_RANDOM_EXISTING:
    raw_parts = sorted(raw_interval_root.rglob("part-*.parquet"))
    assert raw_parts, f"no raw partitions found under: {raw_interval_root}"
    pick = random.choice(raw_parts)
    SYMBOL, YEAR, MONTH = _parse_symbol_year_month(pick)

raw_p = raw_yf_ohlcv_part_path(SYMBOL, INTERVAL, YEAR, MONTH, repo_root=ROOT)
norm_p = norm_ohlcv_part_path(SYMBOL, INTERVAL, YEAR, MONTH, repo_root=ROOT)

print("Selected:", {"symbol": SYMBOL, "year": YEAR, "month": MONTH, "interval": INTERVAL})
print("raw_p :", raw_p)
print("norm_p:", norm_p)

Selected: {'symbol': 'NVDA', 'year': 2022, 'month': 10, 'interval': '1d'}
raw_p : C:\Users\quantbase\Desktop\fineas\data\raw\yfinance\ohlcv\interval=1d\symbol=NVDA\year=2022\month=10\part-2022-10.parquet
norm_p: C:\Users\quantbase\Desktop\fineas\data\norm\ohlcv\interval=1d\symbol=NVDA\year=2022\month=10\part-2022-10.parquet


In [3]:
# Cell 3 — load both partitions
assert raw_p.exists(), f"missing raw partition: {raw_p}"
assert norm_p.exists(), f"missing norm partition: {norm_p}"

raw_df = pd.read_parquet(raw_p)
norm_df = pd.read_parquet(norm_p)

(raw_df.shape, norm_df.shape, raw_df.columns.tolist(), norm_df.columns.tolist())

((21, 10),
 (21, 10),
 ['ts',
  'symbol',
  'open',
  'high',
  'low',
  'close',
  'adj_close',
  'volume',
  'dividends',
  'stock_splits'],
 ['ts',
  'symbol',
  'open',
  'high',
  'low',
  'close',
  'adj_close',
  'volume',
  'returns',
  'movement'])

In [4]:
# Cell 4 — basic QC gates (raw and norm)
qc_raw = qc_raw_ohlcv_partition(raw_df, name=f"qc_raw:{SYMBOL}/{YEAR}-{MONTH:02d}")
qc_norm = qc_norm_ohlcv_partition(norm_df, name=f"qc_norm:{SYMBOL}/{YEAR}-{MONTH:02d}")

qc_raw.to_dict(), qc_norm.to_dict()

({'ok': True,
  'hard_fails': [],
  'warnings': [],
  'stats': {'name': 'qc_raw:NVDA/2022-10',
   'ts_min': '2022-10-03 00:00:00+00:00',
   'ts_max': '2022-10-31 00:00:00+00:00',
   'key_name': 'qc_raw:NVDA/2022-10/key',
   'key_rows': 21,
   'key_key_cols': ['symbol', 'ts'],
   'key_dup_keys_n': 0,
   'time_name': 'qc_raw:NVDA/2022-10/time',
   'time_monotonic_bad_symbols': [],
   'time_max_gap_days': 3.0,
   'time_gap_warn_symbols': [],
   'ohlcv_name': 'qc_raw:NVDA/2022-10/ohlcv',
   'ohlcv_rows': 21,
   'ohlcv_ohlc_violation_rows': 0,
   'ohlcv_volume_negative_rows': 0,
   'ohlcv_nan_counts': {'open': 0,
    'high': 0,
    'low': 0,
    'close': 0,
    'volume': 0}}},
 {'ok': True,
  'hard_fails': [],
  'warnings': ['qc_norm:NVDA/2022-10: returns present where intra-part validation is unavailable (likely cross-part lookback) (n=1)'],
  'stats': {'name': 'qc_norm:NVDA/2022-10',
   'rows': 21,
   'key_name': 'qc_norm:NVDA/2022-10/key',
   'key_rows': 21,
   'key_key_cols': ['symbol',

In [5]:
# Cell 5 — key-set equality check (raw vs norm)
raw_keys = raw_df[["symbol", "ts"]].copy()
norm_keys = norm_df[["symbol", "ts"]].copy()

raw_keys["ts"] = pd.to_datetime(raw_keys["ts"], utc=True)
norm_keys["ts"] = pd.to_datetime(norm_keys["ts"], utc=True)

raw_set = set(map(tuple, raw_keys.to_numpy()))
norm_set = set(map(tuple, norm_keys.to_numpy()))

missing_in_norm = sorted(list(raw_set - norm_set))[:10]
extra_in_norm = sorted(list(norm_set - raw_set))[:10]

assert raw_set == norm_set, {
    "missing_in_norm_sample": missing_in_norm,
    "extra_in_norm_sample": extra_in_norm,
    "raw_n": len(raw_set),
    "norm_n": len(norm_set),
}
print("OK: key sets match exactly.")

OK: key sets match exactly.


In [6]:
# Cell 6 — identity check on base OHLCV fields (raw == norm)
raw_base = raw_df.copy()
norm_base = norm_df.copy()

raw_base["ts"] = pd.to_datetime(raw_base["ts"], utc=True)
norm_base["ts"] = pd.to_datetime(norm_base["ts"], utc=True)

cols = ["open", "high", "low", "close", "adj_close", "volume"]
m = raw_base.merge(norm_base, on=["symbol", "ts"], suffixes=("_raw", "_norm"), how="inner", validate="one_to_one")

for c in cols:
    a = pd.to_numeric(m[f"{c}_raw"], errors="coerce").to_numpy()
    b = pd.to_numeric(m[f"{c}_norm"], errors="coerce").to_numpy()
    # volume can be int; compare exactly if possible, else allclose
    if c == "volume":
        assert np.array_equal(a, b) or np.allclose(a, b, equal_nan=True), f"volume mismatch"
    else:
        assert np.allclose(a, b, equal_nan=True, rtol=0.0, atol=1e-12), f"{c} mismatch"

print("OK: OHLCV identity holds (raw == norm).")

OK: OHLCV identity holds (raw == norm).


In [7]:
# Cell 7 — derived checks: returns and movement recomputation
d = norm_df.copy().sort_values(["symbol", "ts"], kind="mergesort").reset_index(drop=True)
d["ts"] = pd.to_datetime(d["ts"], utc=True)

# recompute returns within partition
d["exp_returns"] = d.groupby("symbol")["adj_close"].pct_change(fill_method=None) * 100.0

# compare returns (ignore first row where exp_returns is NaN)
mask = d["exp_returns"].notna() & d["returns"].notna()
max_abs = (d.loc[mask, "returns"] - d.loc[mask, "exp_returns"]).abs().max()
assert float(max_abs) <= 1e-9, max_abs

def classify(r):
    if pd.isna(r):
        return np.nan
    if r > 0.55:
        return "rise"
    if r < -0.5:
        return "fall"
    return "freeze"

d["exp_movement"] = d["returns"].apply(classify)

# movement should match wherever returns exists
maskm = d["returns"].notna()
mismatch = (d.loc[maskm, "movement"].astype(str) != d.loc[maskm, "exp_movement"].astype(str)).sum()
assert int(mismatch) == 0, int(mismatch)

# when returns is NaN, movement must be NaN
assert d.loc[d["returns"].isna(), "movement"].isna().all(), "movement present where returns is NaN"

print("OK: returns + movement recomputation matches.")

OK: returns + movement recomputation matches.


In [8]:
# Cell 8 — basket-level coverage + movement distribution (filesystem scan)
raw_parts = sorted(raw_interval_root.rglob("part-*.parquet"))
norm_interval_root = ROOT / "data" / "norm" / "ohlcv" / f"interval={INTERVAL}"
norm_parts = sorted(norm_interval_root.rglob("part-*.parquet"))

raw_triplets = [_parse_symbol_year_month(p) for p in raw_parts]
norm_triplets = [_parse_symbol_year_month(p) for p in norm_parts]

raw_set = set(raw_triplets)
norm_set = set(norm_triplets)

missing_norm = sorted(list(raw_set - norm_set))
extra_norm = sorted(list(norm_set - raw_set))

print("raw_parts:", len(raw_parts))
print("norm_parts:", len(norm_parts))
print("missing_norm_parts:", len(missing_norm))
print("extra_norm_parts:", len(extra_norm))

# movement distribution (reads only 'movement' column)
from collections import defaultdict

move_counts = defaultdict(lambda: {"rise": 0, "fall": 0, "freeze": 0, "nan": 0})
for p in norm_parts:
    sym, y, mo = _parse_symbol_year_month(p)
    mv = pd.read_parquet(p, columns=["movement"])["movement"]
    vc = mv.value_counts(dropna=False)
    move_counts[sym]["rise"] += int(vc.get("rise", 0))
    move_counts[sym]["fall"] += int(vc.get("fall", 0))
    move_counts[sym]["freeze"] += int(vc.get("freeze", 0))
    move_counts[sym]["nan"] += int(vc.isna().sum())

pd.DataFrame(move_counts).T.sort_index()

raw_parts: 96
norm_parts: 96
missing_norm_parts: 0
extra_norm_parts: 0


,rise,fall,freeze,nan
AAPL,181,179,140,0
MSFT,188,183,129,0
NEWGEN.NS,185,195,112,0
NVDA,232,211,57,0


In [9]:
# Cell 9 — edge-case probe within chosen month (largest up/down returns)
x = norm_df.copy().sort_values("ts").reset_index(drop=True)

# exclude NaN returns
y = x.dropna(subset=["returns"]).copy()
imax = int(y["returns"].idxmax()) if not y.empty else None
imin = int(y["returns"].idxmin()) if not y.empty else None

def window(df, idx, k=3):
    if idx is None:
        return df.head(0)
    a = max(0, idx - k)
    b = min(len(df), idx + k + 1)
    return df.loc[a:b-1, ["ts", "adj_close", "returns", "movement"]]

print("max-return window")
display(window(x, imax, k=4))

print("min-return window")
display(window(x, imin, k=4))

max-return window


,ts,adj_close,returns,movement
6,2022-10-11 00:00:00+00:00,11.571453,-0.719788,fall
7,2022-10-12 00:00:00+00:00,11.485560,-0.742281,fall
8,2022-10-13 00:00:00+00:00,11.944985,4.000022,rise
9,2022-10-14 00:00:00+00:00,11.212904,-6.128776,fall
10,2022-10-17 00:00:00+00:00,11.873073,5.887580,rise
11,2022-10-18 00:00:00+00:00,11.951975,0.664548,rise
12,2022-10-19 00:00:00+00:00,12.035868,0.701916,rise
13,2022-10-20 00:00:00+00:00,12.178688,1.186623,rise
14,2022-10-21 00:00:00+00:00,12.450347,2.230609,rise


min-return window


,ts,adj_close,returns,movement
0,2022-10-03 00:00:00+00:00,12.496291,3.072759,rise
1,2022-10-04 00:00:00+00:00,13.150466,5.234952,rise
2,2022-10-05 00:00:00+00:00,13.192414,0.318987,freeze
3,2022-10-06 00:00:00+00:00,13.113515,-0.598066,fall
4,2022-10-07 00:00:00+00:00,12.060839,-8.027415,fall
5,2022-10-10 00:00:00+00:00,11.655347,-3.362053,fall
6,2022-10-11 00:00:00+00:00,11.571453,-0.719788,fall
7,2022-10-12 00:00:00+00:00,11.485560,-0.742281,fall
8,2022-10-13 00:00:00+00:00,11.944985,4.000022,rise


In [11]:
# close data ingestion - norm_report Summary check -----

In [13]:
import json

runs_dir = ROOT / "data" / "meta" / "runs"
latest_run = sorted([p for p in runs_dir.glob("run=*") if p.is_dir()])[-1]
report_path = latest_run / "norm_report.json"

print("Using report:", report_path)
payload = json.loads(report_path.read_text(encoding="utf-8"))
summary = payload["summary"]
summary

Using report: C:\Users\quantbase\Desktop\fineas\data\meta\runs\run=20260304T102000Z\norm_report.json


{'run_id': 'run=20260304T102000Z',
 'created_utc': '2026-03-04T11:19:10.595597+00:00',
 'interval': '1d',
 'start': '2022-01-01',
 'end': '2024-01-01',
 'symbols': ['AAPL', 'MSFT', 'NVDA', 'NEWGEN.NS'],
 'months': [[2022, 1],
  [2022, 2],
  [2022, 3],
  [2022, 4],
  [2022, 5],
  [2022, 6],
  [2022, 7],
  [2022, 8],
  [2022, 9],
  [2022, 10],
  [2022, 11],
  [2022, 12],
  [2023, 1],
  [2023, 2],
  [2023, 3],
  [2023, 4],
  [2023, 5],
  [2023, 6],
  [2023, 7],
  [2023, 8],
  [2023, 9],
  [2023, 10],
  [2023, 11],
  [2023, 12]],
 'expected_parts': 96,
 'ok_parts': 96,
 'missing_raw_parts': 0,
 'empty_raw_parts': 0,
 'missing_cols_raw_parts': 0,
 'qc_failed_parts': 0,
 'error_parts': 0,
 'movement_counts_by_symbol': {'AAPL': {'rise': 181,
   'fall': 179,
   'freeze': 140,
   'nan': 2},
  'MSFT': {'rise': 188, 'fall': 183, 'freeze': 129, 'nan': 2},
  'NVDA': {'rise': 232, 'fall': 211, 'freeze': 57, 'nan': 2},
  'NEWGEN.NS': {'rise': 185, 'fall': 195, 'freeze': 112, 'nan': 2}}}

In [15]:
# acceptance assertions

assert summary["expected_parts"] == summary["ok_parts"], summary
assert summary["missing_raw_parts"] == 0, summary
assert summary["empty_raw_parts"] == 0, summary
assert summary["missing_cols_raw_parts"] == 0, summary
assert summary["qc_failed_parts"] == 0, summary
assert summary["error_parts"] == 0, summary

assert len(payload["results"]) == summary["expected_parts"], {
    "results_len": len(payload["results"]),
    "expected_parts": summary["expected_parts"],
}

print("Ingestion loop CLOSEOUT: PASS")

Ingestion loop CLOSEOUT: PASS


In [17]:
# optional summary if failures later

from collections import Counter

status_counts = Counter([x.get("status") for x in payload["results"]])
print("status_counts:", status_counts)

pd.DataFrame(payload["results"]).loc[:, [c for c in ["symbol","year","month","status","rows","out_path"] if c in pd.DataFrame(payload["results"]).columns]].head(20)

status_counts: Counter({'ok': 96})


,symbol,year,month,status,rows,out_path
0,AAPL,2022,1,ok,20,C:\Users\quantbase\Desktop\fineas\data\norm\oh...
1,AAPL,2022,2,ok,19,C:\Users\quantbase\Desktop\fineas\data\norm\oh...
2,AAPL,2022,3,ok,23,C:\Users\quantbase\Desktop\fineas\data\norm\oh...
3,AAPL,2022,4,ok,20,C:\Users\quantbase\Desktop\fineas\data\norm\oh...
4,AAPL,2022,5,ok,21,C:\Users\quantbase\Desktop\fineas\data\norm\oh...
5,AAPL,2022,6,ok,21,C:\Users\quantbase\Desktop\fineas\data\norm\oh...
6,AAPL,2022,7,ok,20,C:\Users\quantbase\Desktop\fineas\data\norm\oh...
7,AAPL,2022,8,ok,23,C:\Users\quantbase\Desktop\fineas\data\norm\oh...
8,AAPL,2022,9,ok,21,C:\Users\quantbase\Desktop\fineas\data\norm\oh...
9,AAPL,2022,10,ok,21,C:\Users\quantbase\Desktop\fineas\data\norm\oh...
